# Tables S3 and S4: MagNET performance across geometries

How accurately MagNET reproduces its DFT training reference (PBE0/pcSseg-1) as geometries move from
stationary to vibrated to solvated, for the foundation model and the chloroform MagNET-x.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/magnet_test_predictions", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import pandas as pd
import magnet_test_predictions_reader
import magnet_benchmark
import paths

In [ ]:
PREDICTIONS = paths.dataset_file("magnet_test_predictions", root=REPO)

def document_path(name):
    # table/spreadsheet outputs go under this notebook's documents/ folder, created on first save
    os.makedirs("documents", exist_ok=True)
    return os.path.join("documents", name)

In [ ]:
rows = magnet_benchmark.exact_stats_table(PREDICTIONS, magnet_test_predictions_reader)
res = pd.DataFrame(rows).set_index(["nucleus", "model", "test_set"])

table_rows = []
for nucleus in ("1H", "13C"):
    for model in magnet_benchmark.MODELS:
        for ts in magnet_benchmark.TEST_SETS:
            r = res.loc[(nucleus, model, ts)]
            pmed, pmae, prmse = magnet_benchmark.PUBLISHED[nucleus][(model, ts)]
            table_rows.append(dict(nucleus=nucleus, model=model, test_set=ts, n=int(r.n),
                                   median_repro=round(r.median_ae, 6), median_SI=round(pmed, 6),
                                   mae_repro=round(r.mae, 6), mae_SI=round(pmae, 6),
                                   rmse_repro=round(r.rmse, 5), rmse_SI=round(prmse, 5)))
table = pd.DataFrame(table_rows)
table_s3 = table[table.nucleus == "1H"].drop(columns="nucleus")
table_s4 = table[table.nucleus == "13C"].drop(columns="nucleus")
print("Table S3 (1H):"); display(table_s3)
print("Table S4 (13C):"); display(table_s4)

# write the reproduced tables to this notebook's documents/ folder, one sheet per SI table
out = document_path("si_table_s03_s04_performance.xlsx")
with pd.ExcelWriter(out) as writer:
    table_s3.to_excel(writer, sheet_name="Table S3 (1H)", index=False)
    table_s4.to_excel(writer, sheet_name="Table S4 (13C)", index=False)
print("wrote", os.path.relpath(out, REPO))

## Exact-reproduction check

In [ ]:
# every row's reproduced median/MAE/RMSE should match the published SI value to a few parts per million
max_dev = (table[["median_repro", "median_SI"]].diff(axis=1).iloc[:, -1].abs().max(),
          table[["mae_repro", "mae_SI"]].diff(axis=1).iloc[:, -1].abs().max(),
          table[["rmse_repro", "rmse_SI"]].diff(axis=1).iloc[:, -1].abs().max())
print("largest reproduced-vs-published deviation (median, MAE, RMSE):", max_dev)
assert max(max_dev) < 1e-3, "a row diverged from the SI by more than float rounding"